In [81]:
import statistics

import pandas as pd

from helpers import *
from regression import *

In [82]:
pd.set_option('display.max_columns', None)

In [83]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
CONFIG = {
    'elo_date': '2025-11-30',  # date for elo ratings from clubelo.com
    'opta_date': '2025-11-28',  # date for elo ratings from the opta -> clubelo regression; elo_date from day X is before the games are played, for opta it depends
    'number_of_sims': 10000,
    'league_id': 106,
    'season': 2025,
    'head_size': 36,
    'country_code_elo': None,  # use this attr. to use elo ratings from clubelo.com
    'country_code_api': 'POL',  # use this attr. to use elo ratings from the opta -> clubelo regression
    'stdev': 0,
    'update_fixtures': False,
    'is_european_league': False,
    'round_no': 18,
}
code = CONFIG['country_code_elo'] if CONFIG['country_code_elo'] is not None else CONFIG['country_code_api']
CONFIG['sorting_order'] = get_sorting_order_for_country_code(code)

In [85]:
# download_elo_data(CONFIG['elo_date'])

In [86]:
# main_regression(**CONFIG)

In [87]:
standings_df = build_historical_standings_table_after_at_most_n_rounds(**CONFIG)
standings_df.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Wisla Plock,1443.38,18,7,9,2,21,12,9,30,0
2,Gornik Zabrze,1513.38,18,9,3,6,29,24,5,30,4
3,Raków Częstochowa,1549.50,17,9,2,6,26,22,4,29,15
4,Jagiellonia,1529.18,16,8,4,4,28,20,8,28,6
5,Cracovia Krakow,1481.76,18,7,6,5,25,21,4,27,12
6,Radomiak Radom,1420.80,18,7,5,6,35,30,5,26,16
7,Lech Poznan,1520.15,17,6,8,3,29,26,3,26,8
8,Zaglebie Lubin,1414.02,17,6,7,4,30,24,6,25,11
9,Korona Kielce,1452.41,18,6,6,6,21,19,2,24,10
10,Pogon Szczecin,1465.96,18,6,3,9,28,32,-4,21,14


In [88]:
CONFIG['update_fixtures'] = False

In [89]:
float(round(standings_df['Points'].sum() / standings_df['Matches played'].sum(), 2))

1.34

In [90]:
sample_season = simulate_season_after_n_rounds(**CONFIG, standings_df=standings_df)
sample_season.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Points,Random order
1,Jagiellonia,1529.18,34,17,8,9,50,34,16,59,12
2,Cracovia Krakow,1481.76,34,17,8,9,47,31,16,59,2
3,Raków Częstochowa,1549.50,34,17,6,11,46,36,10,57,15
4,Wisla Plock,1443.38,34,15,11,8,39,26,13,56,11
5,Legia Warszawa,1497.57,34,15,8,11,42,31,11,53,14
6,Gornik Zabrze,1513.38,34,14,10,10,46,39,7,52,10
7,Pogon Szczecin,1465.96,34,14,6,14,47,45,2,48,3
8,Korona Kielce,1452.41,34,11,14,9,39,33,6,47,4
9,GKS Katowice,1427.57,34,14,5,15,42,43,-1,47,1
10,Lech Poznan,1520.15,34,10,15,9,44,45,-1,45,8


In [91]:
float(round(sample_season['Points'].sum() / sample_season['Matches played'].sum(), 2))

1.35

In [92]:
simulate_odds(**CONFIG, standings_df=standings_df).head(CONFIG['head_size'])

,Home Team,Away Team,Odds H,Odds D,Odds A
0,Lechia Gdansk,Gornik Zabrze,3.45,3.51,2.35
1,Arka Gdynia,Motor Lublin,2.53,3.46,3.17
2,Pogon Szczecin,Radomiak Radom,2.10,3.68,3.98
3,Zaglebie Lubin,Widzew Łódź,2.23,3.58,3.67
4,Piast Gliwice,Legia Warszawa,3.10,3.44,2.58
5,Nieciecza,Jagiellonia,4.19,3.76,2.02
6,Cracovia Krakow,Lech Poznan,2.61,3.44,3.07
7,Raków Częstochowa,GKS Katowice,1.74,4.24,5.25
8,Korona Kielce,Wisla Plock,2.30,3.54,3.53


In [93]:
# full table sim
# results = run_full_table_sims(**CONFIG, standings_df=standings_df)
# results.head(CONFIG['head_size'])

In [94]:
# top 1
winning_places = 1
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:28<00:00, 346.78it/s]

10000 simulations
1 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1549.50,58.29,3190,16,574,31.9,32.1,37.8,3.13,3.12,2.65
2,Jagiellonia,1529.18,57.76,2787,437,116,27.9,32.2,33.4,3.59,3.1,2.99
3,Gornik Zabrze,1513.38,55.32,1394,133,252,13.9,15.3,17.8,7.17,6.55,5.62
4,Lech Poznan,1520.15,53.40,730,145,131,7.3,8.8,10.1,14,11,9.94
5,Wisla Plock,1443.38,51.41,377,126,23,3.8,5.0,5.3,27,20,19
6,Cracovia Krakow,1481.76,50.95,303,92,41,3.0,4.0,4.4,33,25,23
7,Radomiak Radom,1420.80,46.58,47,26,0,0.5,0.7,0.7,213,137,137
8,Zaglebie Lubin,1414.02,45.72,40,30,1,0.4,0.7,0.7,250,143,141
9,Korona Kielce,1452.41,46.09,27,21,2,0.3,0.5,0.5,370,208,200
10,Legia Warszawa,1497.57,45.35,34,19,1,0.3,0.5,0.5,294,189,185


In [95]:
# top 2
winning_places = 2
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:31<00:00, 313.64it/s]

10000 simulations
2 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1549.50,58.33,5528,27,610,55.3,55.6,61.6,1.81,1.8,1.62
2,Jagiellonia,1529.18,57.76,5073,402,224,50.7,54.8,57.0,1.97,1.83,1.75
3,Gornik Zabrze,1513.38,55.25,3116,151,436,31.2,32.7,37.0,3.21,3.06,2.7
4,Lech Poznan,1520.15,53.38,2009,240,213,20.1,22.5,24.6,4.98,4.45,4.06
5,Wisla Plock,1443.38,51.26,1009,260,73,10.1,12.7,13.4,9.91,7.88,7.45
6,Cracovia Krakow,1481.76,51.06,971,202,101,9.7,11.7,12.7,10,8.53,7.85
7,Radomiak Radom,1420.80,46.52,168,79,6,1.7,2.5,2.5,60,40,40
8,Zaglebie Lubin,1414.02,45.79,169,70,6,1.7,2.4,2.4,59,42,41
9,Korona Kielce,1452.41,46.23,154,58,16,1.5,2.1,2.3,65,47,44
10,Legia Warszawa,1497.57,45.36,119,67,1,1.2,1.9,1.9,84,54,53


In [96]:
# top 3
winning_places = 3
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:31<00:00, 317.68it/s]

10000 simulations
3 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1549.50,58.28,7192,16,512,71.9,72.1,77.2,1.39,1.39,1.30
2,Jagiellonia,1529.18,57.81,6710,295,236,67.1,70.0,72.4,1.49,1.43,1.38
3,Gornik Zabrze,1513.38,55.30,4942,109,527,49.4,50.5,55.8,2.02,1.98,1.79
4,Lech Poznan,1520.15,53.40,3518,285,321,35.2,38.0,41.2,2.84,2.63,2.42
5,Wisla Plock,1443.38,51.30,2047,398,126,20.5,24.4,25.7,4.89,4.09,3.89
6,Cracovia Krakow,1481.76,50.84,1763,271,196,17.6,20.3,22.3,5.67,4.92,4.48
7,Radomiak Radom,1420.80,46.61,433,154,22,4.3,5.9,6.1,23,17,16.00
8,Korona Kielce,1452.41,46.17,386,124,25,3.9,5.1,5.4,26,20,19.00
9,Zaglebie Lubin,1414.02,45.80,370,136,13,3.7,5.1,5.2,27,20,19.00
10,Legia Warszawa,1497.57,45.39,323,147,7,3.2,4.7,4.8,31,21,21.00


In [97]:
# top 4
winning_places = 4
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=False)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_top_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:31<00:00, 315.85it/s]

10000 simulations
4 winning places


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Raków Częstochowa,1549.50,58.32,8187,10,404,81.9,82.0,86.0,1.22,1.22,1.16
2,Jagiellonia,1529.18,57.91,7966,205,239,79.7,81.7,84.1,1.26,1.22,1.19
3,Gornik Zabrze,1513.38,55.19,6359,72,533,63.6,64.3,69.6,1.57,1.55,1.44
4,Lech Poznan,1520.15,53.40,4922,257,458,49.2,51.8,56.4,2.03,1.93,1.77
5,Wisla Plock,1443.38,51.34,3278,456,210,32.8,37.3,39.4,3.05,2.68,2.54
6,Cracovia Krakow,1481.76,50.86,3000,373,321,30.0,33.7,36.9,3.33,2.96,2.71
7,Radomiak Radom,1420.80,46.58,876,269,36,8.8,11.5,11.8,11.00,8.73,8.47
8,Korona Kielce,1452.41,46.13,820,215,78,8.2,10.4,11.1,12.00,9.66,8.98
9,Zaglebie Lubin,1414.02,45.82,731,215,35,7.3,9.5,9.8,14.00,11.00,10.00
10,Legia Warszawa,1497.57,45.28,626,219,16,6.3,8.5,8.6,16.00,12.00,12.00


In [98]:
# bottom 3
winning_places = 3
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=True)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_bottom_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:30<00:00, 327.56it/s]

10000 simulations
3 winning places
Reverse: TRUE


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Nieciecza,1341.77,35.73,6081,560,107,60.8,66.4,67.5,1.64,1.51,1.48
2,Piast Gliwice,1402.73,38.77,3713,90,573,37.1,38.0,43.8,2.69,2.63,2.29
3,Lechia Gdansk,1384.67,38.88,3617,39,646,36.2,36.6,43.0,2.76,2.74,2.32
4,Widzew Łódź,1391.44,38.97,3433,201,489,34.3,36.3,41.2,2.91,2.75,2.43
5,Arka Gdynia,1375.64,39.09,3358,706,0,33.6,40.6,40.6,2.98,2.46,2.46
6,Motor Lublin,1398.22,40.20,2714,377,191,27.1,30.9,32.8,3.68,3.24,3.05
7,GKS Katowice,1427.57,41.97,1709,294,190,17.1,20.0,21.9,5.85,4.99,4.56
8,Pogon Szczecin,1465.96,43.85,932,160,163,9.3,10.9,12.6,11,9.16,7.97
9,Legia Warszawa,1497.57,45.35,617,54,153,6.2,6.7,8.2,16,15,12.00
10,Zaglebie Lubin,1414.02,45.78,449,54,141,4.5,5.0,6.4,22,20,16.00


In [99]:
# bottom 1
winning_places = 1
results = run_multiple_sims(**CONFIG, standings_df=standings_df, number_of_winning_places=winning_places, reverse=True)
results.to_csv(f'data/sims/multiple_sims_league_{CONFIG["league_id"]}_bottom_{winning_places}_{datetime.today().strftime("%Y-%m-%d")}.csv', index=False)
results.head(CONFIG['head_size'])

100%|██████████| 10000/10000 [00:30<00:00, 332.97it/s]

10000 simulations
1 winning places
Reverse: TRUE


,Club,Elo,xPts,Wins,Wins in TB,Losses in TB,% Wins,% Wins + won TB,% Wins + all TB,Odds % Wins,Odds % Wins + won TB,Odds % Wins + all TB
1,Nieciecza,1341.77,35.85,2744,464,136,27.4,32.1,33.4,3.64,3.12,2.99
2,Lechia Gdansk,1384.67,38.66,1225,20,395,12.2,12.4,16.4,8.16,8.03,6.1
3,Piast Gliwice,1402.73,38.78,1206,54,334,12.1,12.6,15.9,8.29,7.94,6.27
4,Widzew Łódź,1391.44,38.86,1098,105,244,11.0,12.0,14.5,9.11,8.31,6.91
5,Arka Gdynia,1375.64,39.09,924,370,0,9.2,12.9,12.9,11,7.73,7.73
6,Motor Lublin,1398.22,40.23,702,145,121,7.0,8.5,9.7,14,12,10
7,GKS Katowice,1427.57,42.12,352,77,82,3.5,4.3,5.1,28,23,20
8,Pogon Szczecin,1465.96,43.82,165,43,48,1.6,2.1,2.6,61,48,39
9,Legia Warszawa,1497.57,45.23,88,14,41,0.9,1.0,1.4,114,98,70
10,Zaglebie Lubin,1414.02,45.70,72,18,28,0.7,0.9,1.2,139,111,85
